# 확률론 3주차 과제 3 — 이산 확률변수의 PMF와 CDF

---

**과목**: 확률론  
**주차**: 3주차  
**주제**: 두 주사위 합의 확률질량함수(PMF)와 누적분포함수(CDF)  

---

## 학습 목표

1. 확률변수(Random Variable)의 개념을 이해한다
2. 확률질량함수(PMF)를 구성하고 그 성질을 검증한다
3. 누적분포함수(CDF)를 구성하고 활용법을 익힌다
4. PMF와 CDF의 관계를 이해하고 시각화한다
5. 시뮬레이션을 통해 이론적 분포를 검증한다

## 기본 용어 정리

### 확률변수 (Random Variable)

**확률변수**란 표본공간의 각 원소(근원사건)에 실수를 대응시키는 **함수**이다.

$$X: \Omega \rightarrow \mathbb{R}$$

확률변수는 크게 두 가지로 분류된다:

| 종류 | 정의 | 예시 |
|------|------|------|
| **이산 확률변수** | 셀 수 있는(countable) 값을 가짐 | 주사위 눈금, 동전 앞면 수 |
| **연속 확률변수** | 구간 내의 모든 실수 값을 가짐 | 키, 몸무게, 온도 |

### 확률질량함수 (PMF: Probability Mass Function)

이산 확률변수 $X$가 특정 값 $x$를 가질 확률:

$$f(x) = P(X = x)$$

**PMF의 성질:**
1. $f(x) \geq 0$ (모든 $x$에 대해)
2. $\sum_{\text{모든 } x} f(x) = 1$ (확률의 합은 1)

### 누적분포함수 (CDF: Cumulative Distribution Function)

확률변수 $X$가 $x$ 이하일 확률:

$$F(x) = P(X \leq x) = \sum_{k \leq x} f(k)$$

**CDF의 성질:**
1. $0 \leq F(x) \leq 1$ (확률이므로)
2. **단조증가**: $x_1 < x_2 \Rightarrow F(x_1) \leq F(x_2)$
3. $\lim_{x \to -\infty} F(x) = 0$, $\lim_{x \to +\infty} F(x) = 1$
4. **우연속(right-continuous)**: $F(x) = F(x^+)$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import binom, bernoulli, norm, uniform, expon
from itertools import product
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
sns.set_palette('Set2')

---

## 1단계: 표본공간과 확률변수 정의

### 실험

공정한 주사위 2개를 동시에 던진다.

### 표본공간

$$\Omega = \{(i, j) \mid i, j \in \{1, 2, 3, 4, 5, 6\}\}$$

$$|\Omega| = 6 \times 6 = 36$$

### 확률변수 정의

$$X = \text{두 주사위 눈금의 합} = i + j$$

확률변수 $X$의 **치역(range)**:

$$X \in \{2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12\}$$

In [ ]:
# 표본공간 생성
sample_space = list(product(range(1, 7), range(1, 7)))
print(f"표본공간의 크기: |Omega| = {len(sample_space)}")
print(f"\n표본공간의 처음 10개 원소: {sample_space[:10]}")

# 확률변수 X = 두 주사위의 합
X_values = [d1 + d2 for d1, d2 in sample_space]
X_range = sorted(set(X_values))
print(f"\n확률변수 X의 가능한 값: {X_range}")
print(f"최솟값: {min(X_range)}, 최댓값: {max(X_range)}")

In [ ]:
# 6x6 결과표 — Heatmap 시각화
sum_matrix = np.zeros((6, 6), dtype=int)
for i in range(6):
    for j in range(6):
        sum_matrix[i][j] = (i + 1) + (j + 1)

fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(sum_matrix, annot=True, fmt='d', cmap='RdYlGn_r',
            xticklabels=range(1, 7), yticklabels=range(1, 7),
            linewidths=2, linecolor='white',
            cbar_kws={'label': '두 주사위의 합 (X)'},
            ax=ax, annot_kws={'size': 16, 'weight': 'bold'})

ax.set_xlabel('주사위 2', fontsize=14)
ax.set_ylabel('주사위 1', fontsize=14)
ax.set_title('두 주사위의 합 — 36가지 경우의 수\n'
             '같은 색 = 같은 합 (대각선 방향)', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

### 해석

- 히트맵에서 **같은 합을 가진 셀**이 대각선 방향으로 나란히 위치한다.
- 합이 7인 경우가 가장 많이 나타나며(6개), 합이 2 또는 12인 경우가 가장 적다(각 1개).
- 이 패턴이 PMF의 삼각형 모양을 만들어낸다.

---

## 2단계: 확률질량함수 (PMF) 계산

각 합 $k$에 대해 경우의 수를 세고, 36으로 나누어 확률을 구한다.

$$f(k) = P(X = k) = \frac{\text{합이 } k\text{인 순서쌍의 수}}{36}$$

In [ ]:
# PMF 계산
from collections import Counter

# 각 합의 경우의 수
count_map = Counter(X_values)

# 결과를 정리한 DataFrame
pmf_data = []
for k in X_range:
    count = count_map[k]
    prob = count / 36
    # 해당 합을 만드는 순서쌍 나열
    pairs = [(d1, d2) for d1, d2 in sample_space if d1 + d2 == k]
    pairs_str = ', '.join([f'({d1},{d2})' for d1, d2 in pairs])
    pmf_data.append({
        '합 (k)': k,
        '경우의 수': count,
        '순서쌍': pairs_str,
        'P(X=k)': f'{count}/36',
        '확률값': round(prob, 4)
    })

df_pmf = pd.DataFrame(pmf_data)
print("=" * 80)
print("확률질량함수 (PMF) 표")
print("=" * 80)
print(df_pmf.to_string(index=False))
print(f"\n확률의 합: {df_pmf['확률값'].sum():.4f} (반드시 1이어야 함)")

In [ ]:
# PMF 시각화 — 막대그래프
fig, ax = plt.subplots(figsize=(12, 6))

probs = [count_map[k] / 36 for k in X_range]
counts = [count_map[k] for k in X_range]

# 7을 중심으로 대칭적 색상
cmap = plt.cm.RdYlGn_r
colors = [cmap(0.2 + 0.6 * abs(k - 7) / 5) for k in X_range]

bars = ax.bar(X_range, probs, color=colors, edgecolor='white', linewidth=2, width=0.7)

# 확률값과 분수 표시
for bar, k, c, p in zip(bars, X_range, counts, probs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{c}/36\n({p:.4f})', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xlabel('확률변수 X (두 주사위의 합)', fontsize=14)
ax.set_ylabel('확률 P(X=k)', fontsize=14)
ax.set_title('확률질량함수 (PMF): 두 주사위 합의 확률분포\n'
             '합이 7일 때 확률이 가장 높고, 양 끝으로 갈수록 낮아진다', 
             fontsize=16, fontweight='bold')
ax.set_xticks(X_range)
ax.set_ylim(0, max(probs) + 0.05)

# 대칭 축 표시
ax.axvline(x=7, color='red', linestyle='--', alpha=0.5, label='대칭축 (X=7)')
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

### PMF 성질 검증

**성질 1: 비음수성** — $f(k) \geq 0$ for all $k$

In [ ]:
# PMF 성질 검증
print("=" * 50)
print("PMF 성질 검증")
print("=" * 50)

# 성질 1: 비음수성
all_non_negative = all(p >= 0 for p in probs)
print(f"\n성질 1 (비음수성): 모든 P(X=k) >= 0? → {all_non_negative}")
print(f"  최솟값: {min(probs):.4f}")

# 성질 2: 정규성 (합 = 1)
total_prob = sum(probs)
print(f"\n성질 2 (정규성): ΣP(X=k) = {total_prob:.6f}")
print(f"  1과의 차이: {abs(1 - total_prob):.10f}")

# 대칭성 확인
print(f"\n대칭성 확인: P(X=k) = P(X=14-k)")
for k in range(2, 8):
    k_comp = 14 - k
    p_k = count_map[k] / 36
    p_comp = count_map[k_comp] / 36
    match = "일치" if abs(p_k - p_comp) < 1e-10 else "불일치"
    print(f"  P(X={k}) = {p_k:.4f}, P(X={k_comp}) = {p_comp:.4f} → {match}")

### 해석

- **PMF는 삼각형 모양**: 합이 7에서 최대이고, 양 끝(2, 12)에서 최소이다.
- **완벽한 대칭**: $P(X=k) = P(X=14-k)$. 이는 두 주사위의 역할이 대칭적이기 때문이다.
- **확률의 합 = 1**: PMF의 가장 기본적인 성질이 검증되었다.
- 이 분포는 이산 삼각분포(Discrete Triangular Distribution)의 한 형태이다.

---

## 3단계: 누적분포함수 (CDF) 계산

CDF는 PMF의 **누적합(cumulative sum)**이다:

$$F(x) = P(X \leq x) = \sum_{k \leq x} P(X = k)$$

예를 들어:
- $F(4) = P(X \leq 4) = P(X=2) + P(X=3) + P(X=4)$
- $F(7) = P(X \leq 7) = P(X=2) + P(X=3) + \cdots + P(X=7)$

In [ ]:
# CDF 계산
cdf_values = []
cumulative = 0

cdf_data = []
for k in X_range:
    p_k = count_map[k] / 36
    cumulative += p_k
    cdf_values.append(cumulative)
    cdf_data.append({
        'x': k,
        'P(X=k)': f'{count_map[k]}/36 = {p_k:.4f}',
        'F(x)=P(X<=x)': f'{cumulative:.4f}',
        '분수': f'{sum(count_map[j] for j in range(2, k+1))}/36'
    })

df_cdf = pd.DataFrame(cdf_data)
print("=" * 70)
print("누적분포함수 (CDF) 표")
print("=" * 70)
print(df_cdf.to_string(index=False))

In [ ]:
# PMF와 CDF 동시 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 왼쪽: PMF (막대그래프)
axes[0].bar(X_range, probs, color='#66c2a5', edgecolor='white', linewidth=2, width=0.7, alpha=0.8)
axes[0].set_xlabel('X (두 주사위의 합)', fontsize=13)
axes[0].set_ylabel('P(X = k)', fontsize=13)
axes[0].set_title('확률질량함수 (PMF)', fontsize=16, fontweight='bold')
axes[0].set_xticks(X_range)
axes[0].set_ylim(0, max(probs) + 0.03)

# 오른쪽: CDF (계단함수)
# 계단함수로 CDF 그리기
x_plot = [1.5]  # 시작점 (X=2 이전)
y_plot = [0]

for k, f_val in zip(X_range, cdf_values):
    x_plot.extend([k, k])
    y_plot.extend([y_plot[-1], f_val])  # 점프

x_plot.append(12.5)  # 끝점
y_plot.append(1.0)

axes[1].step(X_range, cdf_values, where='mid', color='#fc8d62', linewidth=2.5, label='CDF')
axes[1].scatter(X_range, cdf_values, color='#fc8d62', s=60, zorder=5)

# 주요 CDF 값 표시
for k, f_val in zip(X_range, cdf_values):
    if k in [2, 7, 12]:  # 주요 값만 레이블
        axes[1].annotate(f'F({k})={f_val:.3f}', xy=(k, f_val),
                        xytext=(k + 0.5, f_val - 0.08),
                        fontsize=10, fontweight='bold',
                        arrowprops=dict(arrowstyle='->', color='gray'))

axes[1].set_xlabel('X (두 주사위의 합)', fontsize=13)
axes[1].set_ylabel('F(x) = P(X ≤ x)', fontsize=13)
axes[1].set_title('누적분포함수 (CDF)', fontsize=16, fontweight='bold')
axes[1].set_xticks(X_range)
axes[1].set_ylim(-0.05, 1.1)
axes[1].axhline(y=0, color='gray', linestyle='-', alpha=0.3)
axes[1].axhline(y=1, color='gray', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.show()

### 해석

- **PMF(왼쪽)**: 각 값에서의 확률을 보여준다. 합이 7에서 가장 높다.
- **CDF(오른쪽)**: 누적 확률을 보여주는 **계단함수(step function)**이다.
  - 이산 확률변수의 CDF는 점프가 있는 계단 형태이다.
  - 각 점프의 크기 = 해당 값의 PMF 값
  - 0에서 시작하여 1에서 끝나는 단조증가 함수이다.

---

## 4단계: CDF를 활용한 확률 계산

CDF를 알면 다양한 확률을 쉽게 계산할 수 있다:

| 확률 유형 | 공식 |
|-----------|------|
| $P(X \leq x)$ | $F(x)$ |
| $P(X > x)$ | $1 - F(x)$ |
| $P(a < X \leq b)$ | $F(b) - F(a)$ |
| $P(X = x)$ | $F(x) - F(x^-)$ (CDF의 점프 크기) |

In [ ]:
# CDF 함수 정의
def cdf_func(x):
    """두 주사위 합의 CDF: F(x) = P(X <= x)"""
    result = 0
    for k in X_range:
        if k <= x:
            result += count_map[k] / 36
    return result

# 다양한 확률 계산
print("=" * 60)
print("CDF를 활용한 확률 계산")
print("=" * 60)

# 질문 1: P(X <= 7)
p_le_7 = cdf_func(7)
print(f"\n질문 1: P(X ≤ 7) = F(7) = {p_le_7:.4f}")
print(f"  의미: 두 주사위 합이 7 이하일 확률")
print(f"  검증: {sum(count_map[k] for k in range(2, 8))}/36 = {sum(count_map[k] for k in range(2, 8))/36:.4f}")

# 질문 2: P(X > 9)
p_gt_9 = 1 - cdf_func(9)
print(f"\n질문 2: P(X > 9) = 1 - F(9) = 1 - {cdf_func(9):.4f} = {p_gt_9:.4f}")
print(f"  의미: 두 주사위 합이 9 초과일 확률")

# 질문 3: P(5 <= X <= 9)
p_5_to_9 = cdf_func(9) - cdf_func(4)  # P(4 < X <= 9)
print(f"\n질문 3: P(5 ≤ X ≤ 9) = F(9) - F(4) = {cdf_func(9):.4f} - {cdf_func(4):.4f} = {p_5_to_9:.4f}")
print(f"  의미: 두 주사위 합이 5 이상 9 이하일 확률")

# 질문 4: P(X = 7)
p_eq_7 = cdf_func(7) - cdf_func(6)
print(f"\n질문 4: P(X = 7) = F(7) - F(6) = {cdf_func(7):.4f} - {cdf_func(6):.4f} = {p_eq_7:.4f}")
print(f"  검증: PMF에서 직접 → {count_map[7]}/36 = {count_map[7]/36:.4f}")

In [ ]:
# CDF 활용 시각화 — P(X <= 7) 영역 표시
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 왼쪽: PMF에서 P(X<=7) 영역
colors_pmf = ['#2ecc71' if k <= 7 else '#e74c3c' for k in X_range]
axes[0].bar(X_range, probs, color=colors_pmf, edgecolor='white', linewidth=2, width=0.7)

# 영역 구분 텍스트
axes[0].axvline(x=7.5, color='black', linestyle='--', linewidth=2, alpha=0.5)
axes[0].text(5, max(probs) * 0.9, f'P(X≤7) = {p_le_7:.4f}', fontsize=13,
            fontweight='bold', color='#2ecc71', ha='center',
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='#2ecc71'))
axes[0].text(10, max(probs) * 0.9, f'P(X>7) = {1-p_le_7:.4f}', fontsize=13,
            fontweight='bold', color='#e74c3c', ha='center',
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='#e74c3c'))

axes[0].set_xlabel('X (두 주사위의 합)', fontsize=13)
axes[0].set_ylabel('P(X = k)', fontsize=13)
axes[0].set_title('PMF에서 P(X≤7) 영역', fontsize=16, fontweight='bold')
axes[0].set_xticks(X_range)

# 오른쪽: CDF에서 P(X<=7) 읽기
axes[1].step(X_range, cdf_values, where='mid', color='#3498db', linewidth=2.5)
axes[1].scatter(X_range, cdf_values, color='#3498db', s=60, zorder=5)

# P(X<=7) 표시
axes[1].axhline(y=p_le_7, color='#2ecc71', linestyle='--', alpha=0.7)
axes[1].axvline(x=7, color='#2ecc71', linestyle='--', alpha=0.7)
axes[1].plot(7, p_le_7, 'o', color='#2ecc71', markersize=12, zorder=10)
axes[1].annotate(f'F(7) = {p_le_7:.4f}', xy=(7, p_le_7),
                xytext=(9, p_le_7 - 0.15), fontsize=13, fontweight='bold', color='#2ecc71',
                arrowprops=dict(arrowstyle='->', color='#2ecc71', lw=2))

axes[1].set_xlabel('X', fontsize=13)
axes[1].set_ylabel('F(x) = P(X ≤ x)', fontsize=13)
axes[1].set_title('CDF에서 P(X≤7) 읽기', fontsize=16, fontweight='bold')
axes[1].set_xticks(X_range)
axes[1].set_ylim(-0.05, 1.1)

plt.tight_layout()
plt.show()

### 해석

- **왼쪽**: PMF에서 $P(X \leq 7)$은 초록색 막대들의 높이를 **모두 더한** 값이다.
- **오른쪽**: CDF에서는 $x=7$에서의 함수값을 **그냥 읽으면** 된다 → 이것이 CDF의 편리함이다.
- CDF를 사용하면 구간 확률도 빼기 한 번으로 구할 수 있다: $P(a < X \leq b) = F(b) - F(a)$

---

## 5단계: CDF 성질 검증

In [ ]:
# CDF 성질 검증
print("=" * 50)
print("CDF 성질 검증")
print("=" * 50)

# 성질 1: 0 <= F(x) <= 1
print(f"\n성질 1: 0 ≤ F(x) ≤ 1")
print(f"  최솟값: F(2) = {cdf_values[0]:.4f} ≥ 0 ✓")
print(f"  최댓값: F(12) = {cdf_values[-1]:.4f} ≤ 1 ✓")

# 성질 2: 단조증가
is_monotone = all(cdf_values[i] <= cdf_values[i+1] for i in range(len(cdf_values)-1))
print(f"\n성질 2: 단조증가? → {is_monotone}")
for i in range(len(X_range) - 1):
    print(f"  F({X_range[i]}) = {cdf_values[i]:.4f} ≤ F({X_range[i+1]}) = {cdf_values[i+1]:.4f} ✓")

# 성질 3: 극한값
print(f"\n성질 3: 극한값")
print(f"  lim(x→-∞) F(x) → 0: F(1) = {cdf_func(1):.4f} (X=1 불가능) ✓")
print(f"  lim(x→+∞) F(x) → 1: F(12) = {cdf_func(12):.4f} ✓")

---

## 6단계: 시뮬레이션을 통한 검증

10,000번 주사위를 던져 **경험적(empirical) PMF**와 이론적 PMF를 비교한다.

In [ ]:
# 시뮬레이션: 10,000번
np.random.seed(42)
n_sim = 10000

die1_sim = np.random.randint(1, 7, size=n_sim)
die2_sim = np.random.randint(1, 7, size=n_sim)
sum_sim = die1_sim + die2_sim

# 경험적 PMF
sim_counter = Counter(sum_sim)
sim_probs = [sim_counter.get(k, 0) / n_sim for k in X_range]

# 비교 표
comparison_data = []
for k, p_th, p_sim in zip(X_range, probs, sim_probs):
    comparison_data.append({
        'X=k': k,
        '이론 PMF': f'{p_th:.4f}',
        '경험 PMF': f'{p_sim:.4f}',
        '오차': f'{abs(p_th - p_sim):.4f}'
    })

df_comp = pd.DataFrame(comparison_data)
print(f"PMF 비교: 이론값 vs 시뮬레이션 ({n_sim:,}회)")
print("=" * 50)
print(df_comp.to_string(index=False))

In [ ]:
# 시뮬레이션 결과 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 왼쪽: PMF 비교 (이론 vs 시뮬레이션)
width = 0.35
x_pos = np.arange(len(X_range))

axes[0].bar(x_pos - width/2, probs, width, label='이론 PMF', color='#3498db', edgecolor='white')
axes[0].bar(x_pos + width/2, sim_probs, width, label=f'경험 PMF (n={n_sim:,})', 
            color='#e67e22', edgecolor='white')
axes[0].set_xlabel('X (두 주사위의 합)', fontsize=13)
axes[0].set_ylabel('확률', fontsize=13)
axes[0].set_title('이론 PMF vs 경험적 PMF (상대빈도)', fontsize=15, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(X_range)
axes[0].legend(fontsize=12)

# 오른쪽: CDF 비교 (이론 vs ECDF)
# 경험적 CDF
ecdf_values = np.cumsum(sim_probs)

axes[1].step(X_range, cdf_values, where='mid', color='#3498db', linewidth=2.5, label='이론 CDF')
axes[1].step(X_range, ecdf_values, where='mid', color='#e67e22', linewidth=2.5, 
             linestyle='--', label=f'경험 CDF (n={n_sim:,})')
axes[1].scatter(X_range, cdf_values, color='#3498db', s=40, zorder=5)
axes[1].scatter(X_range, ecdf_values, color='#e67e22', s=40, zorder=5, marker='s')

axes[1].set_xlabel('X', fontsize=13)
axes[1].set_ylabel('F(x) = P(X ≤ x)', fontsize=13)
axes[1].set_title('이론 CDF vs 경험적 CDF (ECDF)', fontsize=15, fontweight='bold')
axes[1].set_xticks(X_range)
axes[1].set_ylim(-0.05, 1.1)
axes[1].legend(fontsize=12)

plt.tight_layout()
plt.show()

### 해석

- **경험적 PMF**와 이론적 PMF가 매우 근접하다. 10,000회 시행이면 충분히 안정적인 추정치를 얻을 수 있다.
- **ECDF(Empirical CDF)**도 이론적 CDF와 거의 일치한다.
- 시행 횟수를 늘릴수록 두 분포 사이의 차이는 더욱 줄어든다 (큰 수의 법칙).
- **Glivenko-Cantelli 정리**: 경험적 CDF는 표본 크기 $n \to \infty$일 때 이론적 CDF로 균일 수렴한다.

---

## 종합 정리

### PMF와 CDF의 관계

$$\text{PMF} \xrightarrow{\text{누적합}} \text{CDF}$$
$$\text{CDF} \xrightarrow{\text{차분}} \text{PMF}$$

| 항목 | PMF | CDF |
|------|-----|-----|
| **표기** | $f(x) = P(X=x)$ | $F(x) = P(X \leq x)$ |
| **의미** | 각 값에서의 확률 | 누적 확률 |
| **그래프 형태** | 막대그래프 | 계단함수 |
| **합/극한** | $\sum f(x) = 1$ | $F(+\infty) = 1$ |
| **활용** | 특정 값의 확률 | 구간 확률, 분위수 |

### 핵심 개념 요약

1. **확률변수**: 표본공간에서 실수로의 함수. 불확실한 결과를 수치화한다.
2. **PMF**: 이산 확률변수의 확률분포. 각 값에 확률을 부여한다.
3. **CDF**: PMF의 누적합. 구간 확률 계산에 편리하다.
4. **시뮬레이션 검증**: 큰 수의 법칙에 의해 경험적 분포가 이론적 분포에 수렴함을 확인하였다.